# Assignment 1 — Data Acquisition and Source Audit

**Purpose:** Download the official UCI archive, extract all 12 stations, and verify source integrity.

This notebook is an executable evidence view over the shared local pipeline. The
command-line runner and notebook call the same stage implementation, preventing code
drift. Completed artifacts are reused; no Google Colab or Google Drive runtime is used.


In [1]:
# Local WSL project bootstrap. Dependency installation belongs to the
# locked project environment and is intentionally not repeated in notebooks.
from pathlib import Path
import json
import os
import platform
import sys

candidate = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in [candidate, *candidate.parents] if (path / "shared").exists()),
    candidate,
)
if not (PROJECT_ROOT / "shared").exists():
    raise FileNotFoundError(
        "Open this notebook from the local odl-assignment project root."
    )
os.environ["ODL_PROJECT_ROOT"] = str(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from shared.config import PATHS
from shared.runtime import RuntimeOptions
from shared.workflow import run_stage

OPTIONS = RuntimeOptions.from_environment()
print("Project root:", PROJECT_ROOT)
print("Run mode:", OPTIONS.mode)
print("Device policy:", OPTIONS.device)
print("WSL distribution:", os.getenv("WSL_DISTRO_NAME", "not detected"))


I0000 00:00:1786540909.604410    5490 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1786540909.666029    5490 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Project root: /home/nicholas/code/odl-assignment
Run mode: smoke
Device policy: gpu
WSL distribution: Ubuntu-24.04


I0000 00:00:1786540911.040781    5490 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
# Run the shared stage or load its verified completed result.
STAGE_SUMMARY = run_stage("acquire", options=OPTIONS)
STAGE_SUMMARY


{'rows': 420768,
 'columns': 20,
 'stations': 12,
 'start': '2013-03-01 00:00:00',
 'end': '2017-02-28 23:00:00',
 'archive_sha256': 'b04da438b2f331ac0ffd45aebdfec0d20d2367feb5f6948c4b1f7ce1191e33c4',
 'archive_bytes': 8192212,
 'source_url': 'https://archive.ics.uci.edu/static/public/501/beijing%2Bmulti%2Bsite%2Bair%2Bquality%2Bdata.zip',
 'dataset_doi': '10.24432/C5RK5G',
 'development_rows': 359712,
 'frozen_benchmark_rows': 61056,
 'run_mode': 'smoke'}

In [3]:
# Record the exact runtime used for this output when preflight evidence exists.
environment_path = PATHS.environment_artifacts / f"environment_{OPTIONS.mode}.json"
ENVIRONMENT = (
    json.loads(environment_path.read_text(encoding="utf-8"))
    if environment_path.exists()
    else {"status": "preflight evidence not yet generated"}
)
ENVIRONMENT


{'captured_at_utc': '2026-08-12T13:18:29.136112+00:00',
 'project_root': '/home/nicholas/code/odl-assignment',
 'platform': 'Linux-6.6.87.2-microsoft-standard-WSL2-x86_64-with-glibc2.39',
 'system': 'Linux',
 'release': '6.6.87.2-microsoft-standard-WSL2',
 'python': '3.12.3',
 'wsl_distribution': 'Ubuntu-24.04',
 'device': 'gpu',
 'tensorflow_gpu_devices': ['/physical_device:GPU:0'],
 'batch_size': 256,
 'mixed_precision': False,
 'nvidia_smi': {'available': True,
  'name': 'NVIDIA GeForce RTX 3050 Laptop GPU',
  'driver_version': '591.74',
  'memory.total': '4096',
  'memory.used': '665',
  'utilization.gpu': '3',
  'temperature.gpu': '57',
  'power.draw': '18.22'},
 'packages': {'tensorflow': '2.21.0',
  'numpy': '2.3.5',
  'pandas': '2.3.3',
  'pyarrow': '25.0.0',
  'scikit-learn': '1.9.0',
  'matplotlib': '3.11.1',
  'seaborn': '0.13.2'},
 'git': {'available': True,
  'revision': '8cbcae3e1948b87b42012417b61d73ab859f470e',
  'dirty': True},
 'experiment_config': {'methodology_versi

## Why this dataset and target are defensible

**Decision.** Use the official UCI Beijing Multi-Site Air-Quality dataset to predict
next-hour PM2.5. **Alternatives.** Smaller single-station datasets and classification
targets were rejected because they provide less temporal, spatial, and regression
complexity. **Evidence.** The source contains 420,768 hourly rows, 12 stations,
meteorology, co-pollutants, missing runs, and extreme events. **Support.** The UCI DOI
identifies an open, reproducible secondary source
([UCI dataset](https://doi.org/10.24432/C5RK5G)). **Trade-off.** Results describe
historical Beijing monitoring conditions and are predictive, not causal or universally
generalisable.


## Source provenance

The source manifest records the UCI URL, DOI, licence, retrieval timestamp, archive
size, and SHA-256 digest. This makes the downloaded input independently identifiable.


In [4]:
import pandas as pd

source_manifest = json.loads(PATHS.source_manifest.read_text(encoding="utf-8"))
pd.DataFrame([source_manifest])


,url,doi,license,retrieved_at_utc,archive_name,bytes,sha256
0,https://archive.ics.uci.edu/static/public/501/...,10.24432/C5RK5G,CC BY 4.0,2026-08-12T13:18:29.201551+00:00,beijing_multi_site_air_quality.zip,8192212,b04da438b2f331ac0ffd45aebdfec0d20d2367feb5f694...


## Structural integrity

The audit proves the expected 420,768 hourly records, 12 stations, date range, unique
station-time keys, and uninterrupted timestamp grid before modelling begins.


In [5]:
station_audit = pd.read_csv(PATHS.assignment1_tables / "station_source_audit.csv")
raw_schema = pd.read_csv(PATHS.assignment1_tables / "raw_schema.csv")
display(station_audit)
display(raw_schema)


,station,rows,start,end
0,Aotizhongxin,35064,2013-03-01,2017-02-28 23:00:00
1,Changping,35064,2013-03-01,2017-02-28 23:00:00
2,Dingling,35064,2013-03-01,2017-02-28 23:00:00
3,Dongsi,35064,2013-03-01,2017-02-28 23:00:00
4,Guanyuan,35064,2013-03-01,2017-02-28 23:00:00
5,Gucheng,35064,2013-03-01,2017-02-28 23:00:00
6,Huairou,35064,2013-03-01,2017-02-28 23:00:00
7,Nongzhanguan,35064,2013-03-01,2017-02-28 23:00:00
8,Shunyi,35064,2013-03-01,2017-02-28 23:00:00
9,Tiantan,35064,2013-03-01,2017-02-28 23:00:00


,column,dtype,non_null,unique
0,No,int64,420768,35064
1,year,int64,420768,5
2,month,int64,420768,12
3,day,int64,420768,31
4,hour,int64,420768,24
5,PM2.5,float64,412029,888
6,PM10,float64,414319,1084
7,SO2,float64,411747,691
8,NO2,float64,408652,1212
9,CO,float64,400067,132


## Missing measurements before preprocessing

Missing values are reported rather than silently filled during acquisition. Their
treatment is performed causally in the next notebook.


In [6]:
raw_missing = pd.read_csv(PATHS.assignment1_tables / "raw_missing_summary.csv")
missing_by_station = pd.read_csv(
    PATHS.assignment1_tables / "raw_missing_by_station.csv"
)
display(raw_missing)
display(missing_by_station)
print("Acquisition and source audit complete.")


,column,missing_count,missing_percent
0,CO,20701,4.919813
1,O3,13277,3.155421
2,NO2,12116,2.879497
3,SO2,9021,2.143937
4,PM2.5,8739,2.076916
5,PM10,6449,1.532674
6,wd,1822,0.433018
7,DEWP,403,0.095777
8,TEMP,398,0.094589
9,PRES,393,0.093401


,station,PM2.5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,RAIN,WSPM,wd
0,Aotizhongxin,925,718,935,1023,1776,1719,20,20,20,20,14,81
1,Changping,774,582,628,667,1521,604,53,50,53,51,43,140
2,Dingling,779,656,730,1234,2012,1214,53,50,53,51,43,140
3,Dongsi,750,553,663,1601,3197,664,20,20,20,20,14,78
4,Guanyuan,616,429,474,659,1753,1173,20,20,20,20,14,81
5,Gucheng,646,381,507,668,1401,729,51,50,51,43,42,159
6,Huairou,953,777,980,1639,1422,1151,51,53,53,55,49,302
7,Nongzhanguan,628,440,446,692,1206,506,20,20,20,20,14,78
8,Shunyi,913,548,1296,1365,2178,1489,51,51,54,51,44,483
9,Tiantan,677,597,1118,744,1126,843,20,20,20,20,14,78


Acquisition and source audit complete.
